# Hospital Patient Data Analysis

1. Load Dataset & Summary

In [2]:
import pandas as pd

# Load datasets
patient_df = pd.read_csv("Patient_Data.csv")
billing_df = pd.read_csv("Billing_Data.csv")

# Summary
patient_df.info()
patient_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45


Insight:

.info() shows data types, missing values, and structure
Helps identify which columns need cleaning

2. Select Relevant Columns

In [3]:
patient_df = patient_df[['PatientID', 'Department', 'Doctor', 'BillAmount']]

In [4]:
patient_df

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN
5,101,Cardiology,Dr. Smith,5000.0


Keeps only billing-related fields for analysis

3. Drop Administrative Columns

In [5]:
patient_df.drop(['ReceptionistID', 'CheckInTime'], axis=1, inplace=True, errors='ignore')

In [6]:
patient_df

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN
5,101,Cardiology,Dr. Smith,5000.0


errors='ignore' prevents crashing if columns don’t exist

4. Total Bill Amount per Department

In [8]:
dept_bill = patient_df.groupby('Department')['BillAmount'].sum().reset_index()

dept_bill

,Department,BillAmount
0,Cardiology,16200.0
1,Dermatology,0.0
2,Neurology,0.0
3,Orthopedics,7500.0


Use case:

Identify high revenue departments
Helps hospital management decisions

5. Remove Duplicate Patients

In [10]:
patient_df.drop_duplicates(subset='PatientID', keep='first', inplace=True)

Keeps only one record per patient

6. Fill Missing Bill Amount

In [13]:
mean_bill = patient_df['BillAmount'].mean()
patient_df['BillAmount'].fillna(mean_bill, inplace=True)

C:\Users\DELL\AppData\Local\Temp\ipykernel_11756\3084009000.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  patient_df['BillAmount'].fillna(mean_bill, inplace=True)


Ensures:

No missing billing values
Keeps dataset usable for analysis

7. Merge Patient & Billing Dataset

In [14]:
merged_df = pd.merge(patient_df, billing_df, on='PatientID', how='inner')
merged_df

,PatientID,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Cardiology,Dr. Smith,5000.000000,2000,3000
1,102,Neurology,Dr. John,6233.333333,1500,3500
2,103,Orthopedics,Dr. Lee,7500.000000,2500,5000
3,104,Cardiology,Dr. Smith,6200.000000,3000,3200
4,105,Dermatology,Dr. Rose,6233.333333,1000,4000


8. Concatenate New Weekly Patients (Row-wise)

In [17]:
new_patients_df = pd.DataFrame({
    'PatientID': [7, 8, 9],
    'Department': ['Cardiology', 'Neurology', 'Orthopedics'],
    'Doctor': ['Dr. Sharma', 'Dr. Mehta', 'Dr. Singh'],
    'BillAmount': [5000, 7000, 6500]
})

In [18]:
new_patients_df

,PatientID,Department,Doctor,BillAmount
0,7,Cardiology,Dr. Sharma,5000
1,8,Neurology,Dr. Mehta,7000
2,9,Orthopedics,Dr. Singh,6500


In [19]:
final_df = pd.concat([merged_df, new_patients_df], ignore_index=True)

In [20]:
final_df

,PatientID,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Cardiology,Dr. Smith,5000.000000,2000.0,3000.0
1,102,Neurology,Dr. John,6233.333333,1500.0,3500.0
2,103,Orthopedics,Dr. Lee,7500.000000,2500.0,5000.0
3,104,Cardiology,Dr. Smith,6200.000000,3000.0,3200.0
4,105,Dermatology,Dr. Rose,6233.333333,1000.0,4000.0
5,7,Cardiology,Dr. Sharma,5000.000000,NaN,NaN
6,8,Neurology,Dr. Mehta,7000.000000,NaN,NaN
7,9,Orthopedics,Dr. Singh,6500.000000,NaN,NaN


9. Add New Billing Columns (Column-wise)

In [21]:
new_cols = pd.DataFrame({
    'InsuranceCovered': [True]*len(final_df),
    'FinalAmount': final_df['BillAmount'] * 0.9  # Example: 10% discount
})

final_df = pd.concat([final_df, new_cols], axis=1)

In [22]:
final_df 

,PatientID,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount,InsuranceCovered,FinalAmount
0,101,Cardiology,Dr. Smith,5000.000000,2000.0,3000.0,True,4500.0
1,102,Neurology,Dr. John,6233.333333,1500.0,3500.0,True,5610.0
2,103,Orthopedics,Dr. Lee,7500.000000,2500.0,5000.0,True,6750.0
3,104,Cardiology,Dr. Smith,6200.000000,3000.0,3200.0,True,5580.0
4,105,Dermatology,Dr. Rose,6233.333333,1000.0,4000.0,True,5610.0
5,7,Cardiology,Dr. Sharma,5000.000000,NaN,NaN,True,4500.0
6,8,Neurology,Dr. Mehta,7000.000000,NaN,NaN,True,6300.0
7,9,Orthopedics,Dr. Singh,6500.000000,NaN,NaN,True,5850.0


Final Output

In [24]:
final_df.info()
final_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PatientID         8 non-null      int64  
 1   Department        8 non-null      object 
 2   Doctor            8 non-null      object 
 3   BillAmount        8 non-null      float64
 4   InsuranceCovered  5 non-null      float64
 5   FinalAmount       5 non-null      float64
 6   InsuranceCovered  8 non-null      bool   
 7   FinalAmount       8 non-null      float64
dtypes: bool(1), float64(4), int64(1), object(2)
memory usage: 588.0+ bytes


,PatientID,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount,InsuranceCovered,FinalAmount
0,101,Cardiology,Dr. Smith,5000.000000,2000.0,3000.0,True,4500.0
1,102,Neurology,Dr. John,6233.333333,1500.0,3500.0,True,5610.0
2,103,Orthopedics,Dr. Lee,7500.000000,2500.0,5000.0,True,6750.0
3,104,Cardiology,Dr. Smith,6200.000000,3000.0,3200.0,True,5580.0
4,105,Dermatology,Dr. Rose,6233.333333,1000.0,4000.0,True,5610.0
